In [2]:
import pandas as pd
import numpy as np
import torch
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import jax
import jax.numpy as jnp
import optax
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

In [5]:
data = fetch_california_housing(as_frame = True)
df = data.frame

In [8]:
df.columns

Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude', 'MedHouseVal'],
      dtype='object')

In [15]:
X = df.drop(columns=['MedHouseVal']).values.astype(np.float32)
y = df['MedHouseVal'].values.astype(np.float32).reshape(-1,1)

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X,y)

X_mean, X_std = X_train.mean(axis=0), X_train.std(axis=0)

X_train_s = (X_train - X_mean)/X_std

X_test_s = (X_test - X_mean)/ X_std

In [22]:
X_train_t = torch.from_numpy(X_train_s)
y_train_t = torch.from_numpy(y_train)
X_test = torch.from_numpy(X_test_s)
y_test = torch.from_numpy(y_test)

In [27]:
torch.manual_seed(42)
n_features = X_train_s.shape[1]

In [29]:
model = torch.nn.Linear(in_features=n_features, out_features=1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = torch.nn.MSELoss()

epochs = 200
for epoch in range(epochs):
    y_pred = model(X_train_t)
    loss = loss_fn(y_pred, y_train_t)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"epoch {epoch:4d} | train MSE {loss.item():.4f}")

epoch    0 | train MSE 6.0985
epoch  100 | train MSE 0.5238
